In [1]:
import os
from dotenv import load_dotenv

print("loaded:", load_dotenv(r"API.env"))                          # True = a .env was found and read
print("key present:", bool(os.environ.get("GEMINI_API_KEY")))  # True = the key is present in the environment

loaded: True
key present: True


# implementing phase A: replicate the technical summary generation of LAsset

## module extraction to feed into llm

In [2]:
!python stage_a.py "LAsset-Security-Assets\SoC\\neorv32 RTL"

Extracted 51 IP modules from LAsset-Security-Assets\SoC\\neorv32 RTL
Saved -> stage_a_out/ip_modules.json, list_names.txt, list_enriched.txt


In [2]:
# Cell 1 — (run once) extract + save the two lists
from stage_a import extract_ip_modules, save_lists


mods = extract_ip_modules(r"LAsset-Security-Assets\SoC\\neorv32 RTL")
print("saved ->", save_lists(mods), f"({len(mods)} modules)")

saved -> stage_a_out (51 modules)


In [2]:
# Cell 2 — load modules + prompt
from stage_a import Module, render_modules, load_modules
from prompts import BATCH_SYSTEM

mods = load_modules("stage_a_out/ip_modules.json")
ENRICH = False          # flip to False for the name-only ablation

## Extracted module name fade into llm to keep/drop

In [ ]:
# Cell 3 — the client (free-tier Gemini; GEMINI_API_KEY in env)
import os
import time, random
from google import genai
from google.genai import types

class GeminiClient:
    def __init__(self, model="gemini-3-flash-preview", temperature=0.0, max_retries=6):
        self.c, self.model, self.t, self.r = genai.Client(api_key=os.environ.get("GEMINI_API_KEY")), model, temperature, max_retries

    def complete(self, system, user):
        wait = 2.0
        for a in range(self.r):
            try:
                return self.c.models.generate_content(
                    model=self.model, contents=user,
                    config=types.GenerateContentConfig(
                        system_instruction=system, temperature=self.t,
                        response_mime_type="application/json")).text
            except Exception:
                if a == self.r - 1: raise
                time.sleep(wait + random.uniform(0, 1)); wait *= 2
                
client = GeminiClient()

In [4]:
# Cell 4 — one batched call + map decisions back by name
import re, json

def parse_json_array(text):
    t = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M).strip()
    i, j = t.find("["), t.rfind("]")
    if i != -1 and j != -1 and j > i: t = t[i:j+1]
    try:
        a = json.loads(t); return a if isinstance(a, list) else []
    except Exception:
        return []

def classify_batch(mods, client, system, enrich, min_conf_drop=0.5):
    user = "Classify each IP module below. Return the JSON array only.\n\nModules:\n" + render_modules(mods, enrich)
    arr = parse_json_array(client.complete(system, user))
    by = {str(o["name"]).strip().lower(): o for o in arr if isinstance(o, dict) and "name" in o}
    for m in mods:
        o = by.get(m.name.lower())
        if o is None:
            m.keep, m.cia, m.confidence = True, [], 0.0
            m.rationale = "[no decision returned; kept for recall]"
        else:
            m.family = str(o.get("family", "")).strip()      # <-- ADD THIS
            m.keep = bool(o.get("keep", True)); m.cia = list(o.get("cia", []) or [])
            m.confidence = float(o.get("confidence", 0.0) or 0.0); m.rationale = str(o.get("rationale", "")).strip()
        if m.keep is False and m.confidence < min_conf_drop:
            m.keep = True; m.rationale += "  [override: low-confidence drop -> kept]"
    return {"returned": len(arr), "expected": len(mods)}

info = classify_batch(mods, client, BATCH_SYSTEM, enrich=ENRICH)
print("returned", info["returned"], "/", info["expected"])

returned 51 / 51


In [6]:
# Cell 5 — save + eyeball

tag = "enriched" if ENRICH else "names"
json.dump([m.to_dict() for m in mods], open(f"stage_a_out/keep_drop_{tag}.json", "w"), indent=2)

kept, dropped = [m for m in mods if m.keep], [m for m in mods if not m.keep]
print(f"KEPT {len(kept)} | DROPPED {len(dropped)}")
for m in dropped: print("  drop:", m.name, "::", m.rationale)

KEPT 48 | DROPPED 3
  drop: neorv32_application_image :: Pre-initialized program memory image blob, not hardware logic.
  drop: neorv32_bootloader_image :: Pre-initialized bootloader memory image blob, not hardware logic.
  drop: neorv32_package :: VHDL package containing constants/types, no synthesizable logic.


In [7]:
import json

# ------------------------------------------------------------------
# File paths
# ------------------------------------------------------------------
initial_asset_file = r"LAsset-Security-Assets\SoC\RTL\asset_list_neorv32_initial.json"
refined_asset_file = r"LAsset-Security-Assets\SoC\Spec.+RTL\asset_list_neorv32_refined.json"
keep_drop_file = r"stage_a_out\keep_drop_names.json"

# ------------------------------------------------------------------
# Load JSON files
# ------------------------------------------------------------------
with open(initial_asset_file, "r", encoding="utf-8") as f:
    initial_assets = json.load(f)

with open(refined_asset_file, "r", encoding="utf-8") as f:
    text = f.read()

decoder = json.JSONDecoder()

refined_assets = []
idx = 0

while idx < len(text):
    # Skip whitespace
    while idx < len(text) and text[idx].isspace():
        idx += 1

    if idx >= len(text):
        break

    obj, end = decoder.raw_decode(text, idx)
    refined_assets.append(obj)
    idx = end

print(f"Loaded {len(refined_assets)} JSON objects.")

with open(keep_drop_file, "r", encoding="utf-8") as f:
    keep_drop = json.load(f)

# ------------------------------------------------------------------
# Collect IP names
# ------------------------------------------------------------------
initial_ips = {
    entry["IP"]
    for entry in initial_assets
    if "IP" in entry
}

refined_ips = {
    entry["IP"]
    for entry in refined_assets
    if "IP" in entry
}

# ------------------------------------------------------------------
# Collect only modules marked keep=True
# ------------------------------------------------------------------
kept_modules = {
    entry["name"]
    for entry in keep_drop
    if entry.get("keep", False)
}

# ------------------------------------------------------------------
# Compare against keep=True modules
# ------------------------------------------------------------------
missing_initial = sorted(initial_ips - kept_modules)
missing_refined = sorted(refined_ips - kept_modules)

# ------------------------------------------------------------------
# Compare the two asset files
# ------------------------------------------------------------------
initial_only = sorted(initial_ips - refined_ips)
refined_only = sorted(refined_ips - initial_ips)

# ------------------------------------------------------------------
# Print results
# ------------------------------------------------------------------
print("=" * 70)
print(f"Initial asset IPs      : {len(initial_ips)}")
print(f"Refined asset IPs      : {len(refined_ips)}")
print(f"keep=True module names : {len(kept_modules)}")
print("=" * 70)

print("\n[1] Initial asset IPs missing from keep=True modules")
if missing_initial:
    for m in missing_initial:
        print("  ", m)
else:
    print("  None")

print("\n[2] Refined asset IPs missing from keep=True modules")
if missing_refined:
    for m in missing_refined:
        print("  ", m)
else:
    print("  None")

print("\n[3] IPs present only in initial asset file")
if initial_only:
    for m in initial_only:
        print("  ", m)
else:
    print("  None")

print("\n[4] IPs present only in refined asset file")
if refined_only:
    for m in refined_only:
        print("  ", m)
else:
    print("  None")

Loaded 41 JSON objects.
Initial asset IPs      : 41
Refined asset IPs      : 41
keep=True module names : 48

[1] Initial asset IPs missing from keep=True modules
  None

[2] Refined asset IPs missing from keep=True modules
  None

[3] IPs present only in initial asset file
  None

[4] IPs present only in refined asset file
  None


## Technical Summary generation from the Spec

In [4]:
# Cell R1 — RAG corpus: chunk the datasheet .adoc sources, embed once, cache
from pathlib import Path
import json, hashlib
import numpy as np

SPEC_DIR   = Path(r"neorv32/docs/datasheet")     # pinned at b8a37e7f~1
CACHE_DIR  = Path("stage_a_out/rag_cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
CHUNK_SIZE = 1000    # paper: 1000-char chunks
CHUNK_OVLP = 200     # paper: 200-char overlap
EMB_MODEL  = "BAAI/bge-small-en-v1.5"

def load_corpus(spec_dir):
    docs = [(p.name, p.read_text(encoding="utf-8", errors="ignore"))
            for p in sorted(spec_dir.glob("*.adoc"))]
    if not docs:
        raise FileNotFoundError(f"no .adoc files under {spec_dir.resolve()}")
    return docs

def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVLP):
    step = size - overlap
    return [text[i:i+size] for i in range(0, max(len(text) - overlap, 1), step)]

def build_chunks(docs):
    # chunk per source file, never across file boundaries (attribution integrity)
    return [{"id": f"{src}#{j}", "source": src, "text": c}
            for src, text in docs for j, c in enumerate(chunk_text(text))]

docs   = load_corpus(SPEC_DIR)
chunks = build_chunks(docs)
fp     = hashlib.sha256("".join(c["text"] for c in chunks).encode()).hexdigest()[:16]
emb_f  = CACHE_DIR / f"emb_{fp}.npy"
(CACHE_DIR / f"chunks_{fp}.jsonl").write_text(
    "\n".join(json.dumps(c) for c in chunks), encoding="utf-8")
print(f"{len(docs)} adoc files -> {len(chunks)} chunks | fingerprint {fp}")

if emb_f.exists():
    embs = np.load(emb_f); print("loaded cached embeddings:", embs.shape)
else:
    from sentence_transformers import SentenceTransformer
    _m = SentenceTransformer(EMB_MODEL)          # first run downloads ~130 MB
    embs = np.asarray(_m.encode([c["text"] for c in chunks], batch_size=64,
                                show_progress_bar=True, normalize_embeddings=True),
                      dtype=np.float32)
    np.save(emb_f, embs); print("embedded + cached:", embs.shape)

41 adoc files -> 654 chunks | fingerprint 65e26791466b757d
loaded cached embeddings: (654, 384)


In [5]:
# Cell R2 — index + retrieval (FAISS per paper; numpy fallback identical at this scale)
TOP_K = 20   # paper: top 20 chunks per query

try:
    import faiss
    _index = faiss.IndexFlatIP(embs.shape[1]); _index.add(embs)
    def _search(qv, k):
        s, i = _index.search(qv[None, :], k); return s[0], i[0]
    print("FAISS index:", _index.ntotal)
except ImportError:
    print("faiss unavailable -> numpy cosine fallback")
    def _search(qv, k):
        sims = embs @ qv; idx = np.argsort(-sims)[:k]; return sims[idx], idx

_qmodel = None
def embed_query(q):
    global _qmodel
    if _qmodel is None:
        from sentence_transformers import SentenceTransformer
        _qmodel = SentenceTransformer(EMB_MODEL)
    return np.asarray(_qmodel.encode([q], normalize_embeddings=True), dtype=np.float32)[0]

def retrieve(query, k=TOP_K):
    scores, idx = _search(embed_query(query), k)
    return [{"score": float(s), **chunks[i]} for s, i in zip(scores, idx)]

FAISS index: 654


In [14]:
# Cell R3 — retrieval sanity check: positive + negative control
def build_rag_query(mod):
    return (f"{mod.name} {' '.join(mod.units)} "
            f"function registers configuration generics interrupts "
            f"access control privilege security")

for name in ("neorv32_trng", "neorv32_fifo"):
    m = next(x for x in mods if x.name == name)
    print(f"\n=== {name} ===")
    for r in retrieve(build_rag_query(m), k=5):
        print(f"{r['score']:.3f}  {r['id']:28s} {r['text'][:90].replace(chr(10),' ')}")


=== neorv32_trng ===


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

0.768  software.adoc#3                  | Description | -                   | `neorv32.h`            | Main NEORV32 library fi
0.761  software_rte.adoc#9          /NEORV32-RTE>\n <5> ---- <1> Illegal 32-bit instruction `MTINST=0x000000FF` at address `PC
0.760  cpu_isa.adoc#8               r module extension (hardware breakpoint)         | <<_processor_top_entity_generics, `OCD_
0.748  cpu_isa.adoc#2               ific / NEORV32-specific extension               | Always enabled | <<_zaamo_isa_extension,
0.747  soc_trng.adoc#0              <<< :sectnums: ==== True Random-Number Generator (TRNG)  [cols="<3,<3,<4"] [grid="none"] |

=== neorv32_fifo ===
0.772  soc_sysinfo.adoc#0           <<< :sectnums: ==== System Configuration Information Memory (SYSINFO)  [cols="<3,<3,<4"] [
0.771  cpu_isa.adoc#8               r module extension (hardware breakpoint)         | <<_processor_top_entity_generics, `OCD_
0.767  cpu_isa.adoc#19              ess-privileged** operation mode. Code executed in use

In [15]:
# Cell R4 — retrieval eval: A/B query construction × bge query prefix
BGE_PREFIX = "Represent this sentence for searching relevant passages: "

def embed_query(q, prefix=False):
    global _qmodel
    if _qmodel is None:
        from sentence_transformers import SentenceTransformer
        _qmodel = SentenceTransformer(EMB_MODEL)
    return np.asarray(_qmodel.encode([BGE_PREFIX + q if prefix else q],
                                     normalize_embeddings=True), dtype=np.float32)[0]

def retrieve_x(query, k=TOP_K, prefix=False):
    scores, idx = _search(embed_query(query, prefix), k)
    return [{"score": float(s), **chunks[i]} for s, i in zip(scores, idx)]

def q_v0(m):   # current keyword-soup query (baseline)
    return (f"{m.name} {' '.join(m.units)} function registers configuration "
            f"generics interrupts access control privilege security")

def q_v1(m):   # module-specific: name + units + Stage-A header, no generic tail
    h = f" -- {m.header}" if m.header else ""
    return f"{m.name} {' '.join(m.units)}{h}"

srcs = {c["source"] for c in chunks}
def expected_adoc(name):
    cand = f"soc_{name.replace('neorv32_','')}.adoc"
    return cand if cand in srcs else None

evalable = [m for m in mods if expected_adoc(m.name)]
print(f"evaluating on {len(evalable)} modules with a dedicated soc_*.adoc\n")

for label, qfn, pfx in [("v0  no-prefix", q_v0, False),
                        ("v0  +prefix ", q_v0, True),
                        ("v1  no-prefix", q_v1, False),
                        ("v1  +prefix ", q_v1, True)]:
    ranks = []
    for m in evalable:
        exp  = expected_adoc(m.name)
        hits = retrieve_x(qfn(m), k=TOP_K, prefix=pfx)
        r = next((i for i, h in enumerate(hits, 1) if h["source"] == exp), None)
        ranks.append(r)
    hit1  = sum(r == 1 for r in ranks if r)
    hit5  = sum(r is not None and r <= 5 for r in ranks)
    miss  = sum(r is None for r in ranks)
    found = [r for r in ranks if r]
    print(f"{label} | hit@1 {hit1:2d}/{len(ranks)} | hit@5 {hit5:2d}/{len(ranks)} "
          f"| miss@20 {miss} | mean rank {sum(found)/len(found):.1f}")

evaluating on 22 modules with a dedicated soc_*.adoc

v0  no-prefix | hit@1 11/22 | hit@5 18/22 | miss@20 1 | mean rank 2.6
v0  +prefix  | hit@1 11/22 | hit@5 19/22 | miss@20 1 | mean rank 2.4
v1  no-prefix | hit@1 22/22 | hit@5 22/22 | miss@20 0 | mean rank 1.0
v1  +prefix  | hit@1 22/22 | hit@5 22/22 | miss@20 0 | mean rank 1.0


## summarization

In [6]:
from rag import build_index, retrieve, build_rag_query
chunks, embs = build_index(spec_dir=r"neorv32/docs/datasheet")

rag: 654 chunks | emb (654, 384) | cache c78ed0005030067b


In [7]:
# Cell S1 — text-mode completion (your client hardcodes JSON mime; summaries are markdown)
def complete_text(client, system, user, max_tokens=2048):
    import time, random
    from google.genai import types
    wait = 2.0
    for a in range(client.r):
        try:
            return client.c.models.generate_content(
                model=client.model, contents=user,
                config=types.GenerateContentConfig(
                    system_instruction=system, temperature=client.t,
                    max_output_tokens=max_tokens)).text   # no JSON mime here
        except Exception:
            if a == client.r - 1: raise
            time.sleep(wait + random.uniform(0, 1)); wait *= 2

In [8]:
import prompts, importlib
importlib.reload(prompts)
from prompts import SPEC_SUMMARY_SYSTEM

In [9]:
from stage_a import load_modules

mods = load_modules("stage_a_out/keep_drop_names.json")   # your renamed file
kept = [m for m in mods if m.keep]
print(len(kept))   # must print 48 — don't run S2 until it does

48


In [10]:
print(len(mods), sum(1 for m in mods if m.keep is True),
      sum(1 for m in mods if m.keep is None))

51 48 0


In [ ]:
# Cell S2 — Task 2: Modular Spec Summarization (resumable; one .md per kept module)
from prompts import SPEC_SUMMARY_SYSTEM   # add it to prompts.py alongside the batch prompts
import time, json
from pathlib import Path

SUM_DIR = Path("stage_a_out/summaries"); SUM_DIR.mkdir(parents=True, exist_ok=True)

def assemble_context(hits):
    return "\n\n".join(f"[{h['id']}]\n{h['text']}" for h in hits)

kept = [m for m in mods if m.keep]
done = skipped = failed = 0

for m in kept:
    out_f = SUM_DIR / f"{m.name}.md"
    if out_f.exists():
        skipped += 1; continue
    hits = retrieve(build_rag_query(m), k=TOP_K)
    user = (f"TARGET IP MODULE: {m.name}\n"
            f"Design units inside this module: {', '.join(m.units)}\n\n"
            f"CONTEXT (retrieved specification excerpts):\n{assemble_context(hits)}")
    try:
        text = complete_text(client, SPEC_SUMMARY_SYSTEM, user)
        out_f.write_text(text or "", encoding="utf-8")
        (SUM_DIR / f"{m.name}.retrieval.json").write_text(json.dumps(
            {"model": client.model,                                   # <- the model field
            "chunks": [{"id": h["id"], "score": h["score"]} for h in hits]}, indent=1))
        done += 1
        print(f"ok   {m.name}")
    except Exception as e:
        failed += 1
        print(f"FAIL {m.name}: {e}")
    time.sleep(1.5)   # be gentle with free-tier RPM

print(f"\nsummaries: {done} new | {skipped} skipped (exist) | {failed} failed -- rerun cell to resume")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ok   neorv32_cpu_cp_cfu
ok   neorv32_cpu_cp_cond
ok   neorv32_cpu_cp_crypto
ok   neorv32_cpu_cp_fpu
ok   neorv32_cpu_cp_muldiv
ok   neorv32_cpu_cp_shifter
ok   neorv32_cpu_decompressor
ok   neorv32_cpu_frontend
ok   neorv32_cpu_icc
ok   neorv32_cpu_lsu
ok   neorv32_cpu_pmp
ok   neorv32_cpu_regfile
ok   neorv32_crc
ok   neorv32_debug_auth
ok   neorv32_debug_dm
ok   neorv32_debug_dtm
ok   neorv32_dma
ok   neorv32_dmem
ok   neorv32_fifo
ok   neorv32_gpio
ok   neorv32_gptmr
FAIL neorv32_hwspinlock: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3-flash\nPlease retry in 48.802043959s.', 'status': 'RESOURCE_EXHAUSTED',

In [13]:
# Cell R4 — retrieval regression eval (run after build_index; re-run whenever
# EMB_MODEL, chunk params, corpus version, or build_rag_query change)
import rag
from rag import retrieve, build_rag_query, TOP_K

assert rag._search is not None, "call build_index() first"

BGE_PREFIX = "Represent this sentence for searching relevant passages: "

def q_v0(m):   # keyword-soup baseline (kept for the record)
    return (f"{m.name} {' '.join(m.units)} function registers configuration "
            f"generics interrupts access control privilege security")

variants = [
    ("v0 no-prefix            ", q_v0),
    ("v0 +prefix              ", lambda m: BGE_PREFIX + q_v0(m)),
    ("v1 no-prefix (PRODUCTION)", build_rag_query),
    ("v1 +prefix              ", lambda m: BGE_PREFIX + build_rag_query(m)),
]

srcs = {c["source"] for c in rag._chunks}
def expected_adoc(name):
    cand = f"soc_{name.replace('neorv32_', '')}.adoc"
    return cand if cand in srcs else None

evalable = [m for m in mods if expected_adoc(m.name)]
print(f"evaluating on {len(evalable)} modules with a dedicated soc_*.adoc\n")

for label, qfn in variants:
    ranks = []
    for m in evalable:
        hits = retrieve(qfn(m), k=TOP_K)
        exp  = expected_adoc(m.name)
        ranks.append(next((i for i, h in enumerate(hits, 1)
                           if h["source"] == exp), None))
    hit1  = sum(r == 1 for r in ranks if r)
    hit5  = sum(r is not None and r <= 5 for r in ranks)
    miss  = sum(r is None for r in ranks)
    found = [r for r in ranks if r]
    print(f"{label} | hit@1 {hit1:2d}/{len(ranks)} | hit@5 {hit5:2d}/{len(ranks)} "
          f"| miss@20 {miss} | mean rank {sum(found)/len(found):.1f}")

evaluating on 22 modules with a dedicated soc_*.adoc

v0 no-prefix             | hit@1 11/22 | hit@5 18/22 | miss@20 1 | mean rank 2.6
v0 +prefix               | hit@1 11/22 | hit@5 19/22 | miss@20 1 | mean rank 2.4
v1 no-prefix (PRODUCTION) | hit@1 22/22 | hit@5 22/22 | miss@20 0 | mean rank 1.0
v1 +prefix               | hit@1 22/22 | hit@5 22/22 | miss@20 0 | mean rank 1.0


## RTL parsing

In [39]:

import rtl_parse, importlib
importlib.reload(rtl_parse)
from rtl_parse import parse_rtl_file

In [28]:
from rtl_parse import parse_rtl_file

parse_rtl_file(r"LAsset-Security-Assets\SoC\neorv32 RTL\neorv32_cpu_pmp.vhd")

{'file': 'neorv32_cpu_pmp',
 'entities': [{'entity': 'neorv32_cpu_pmp',
   'ports': [{'name': 'clk_i', 'dir': 'in', 'type': 'std_ulogic'},
    {'name': 'rstn_i', 'dir': 'in', 'type': 'std_ulogic'},
    {'name': 'ctrl_i', 'dir': 'in', 'type': 'ctrl_bus_t'},
    {'name': 'csr_o',
     'dir': 'out',
     'type': 'std_ulogic_vector(XLEN-1 downto 0)'},
    {'name': 'addr_ls_i',
     'dir': 'in',
     'type': 'std_ulogic_vector(XLEN-1 downto 0)'},
    {'name': 'fault_o', 'dir': 'out', 'type': 'std_ulogic'}],
   'signals': [{'name': 'pmpcfg', 'type': 'pmpcfg_t'},
    {'name': 'pmpcfg_we', 'type': 'std_ulogic_vector(3 downto 0)'},
    {'name': 'pmpaddr', 'type': 'pmpaddr_t'},
    {'name': 'pmpaddr_we', 'type': 'std_ulogic_vector(15 downto 0)'},
    {'name': 'cfg_rd', 'type': 'csr_cfg_rd_t'},
    {'name': 'cfg_rd32', 'type': 'csr_cfg_rd32_t'},
    {'name': 'addr_rd', 'type': 'csr_addr_rd_t'},
    {'name': 'acc_addr', 'type': 'std_ulogic_vector(XLEN-1 downto 0)'},
    {'name': 'acc_priv', 'type'

In [29]:
RTL_ROOT = Path(r"LAsset-Security-Assets\SoC\neorv32 RTL")

for m in kept:
    r = parse_rtl_file(RTL_ROOT / m.file)
    for e in r["entities"]:
        if not e["ports"]:
            print(f"{m.name}/{e['entity']}: NO PORTS")
        for p in e["ports"]:
            if "(" in p["type"] and not p["type"].rstrip().endswith(")"):
                print(f"{m.name}/{e['entity']}/{p['name']}: bad type {p['type']!r}")

In [40]:
def merge(ann, det):
    """LLM supplies function/kind; deterministic supplies name+type (immutable)."""
    by = {a.get("name"): a for a in ann}
    out = []
    for d in det:
        a = by.get(d["name"], {})
        out.append({**d,                                   # name + true VHDL type
                    "kind": a.get("kind", ""),
                    "function": a.get("function", "function unclear from RTL")})
    return out

In [44]:
# Cell P1 — Task 3: per-entity RTL parsing (deterministic extract + LLM annotate)
from llm import parse_json_array, complete_text
from rtl_parse import parse_rtl_file
from prompts import RTL_PORT_PARSER_SYSTEM, RTL_SIGNAL_PARSER_SYSTEM
import json, time
from pathlib import Path

PARSE_DIR = Path("stage_a_out/rtl_parsed"); PARSE_DIR.mkdir(parents=True, exist_ok=True)
RTL_ROOT  = Path(r"LAsset-Security-Assets/SoC/neorv32 RTL")

def annotate(client, system, elements, rtl_text):
    if not elements: return []
    user = (f"RTL:\n{rtl_text}\n\nElements to annotate "
            f"(return EXACTLY these, same names, same order):\n{json.dumps(elements, indent=1)}")
    return parse_json_array(complete_text(client, system, user, max_tokens=4096))

def guard(got, exp, det):
    """count-check; fall back to deterministic list (names/types) if LLM dropped/added."""
    return got if len(got) == len(exp) else det

kept = [m for m in mods if m.keep]
for m in kept:
    out_f = PARSE_DIR / f"{m.name}.json"
    if out_f.exists(): continue
    rtl = (RTL_ROOT / m.file).read_text(encoding="utf-8", errors="ignore")
    parsed = parse_rtl_file(RTL_ROOT / m.file)
    try:
        result_entities, warned = [], False
        for ent in parsed["entities"]:
            p_ann = annotate(client, RTL_PORT_PARSER_SYSTEM,   ent["ports"],   rtl); time.sleep(7)
            s_ann = annotate(client, RTL_SIGNAL_PARSER_SYSTEM, ent["signals"], rtl); time.sleep(7)
            if len(p_ann) != len(ent["ports"]) or len(s_ann) != len(ent["signals"]):
                warned = True
                print(f"  WARN {m.name}/{ent['entity']}: "
                      f"p {len(p_ann)}/{len(ent['ports'])} s {len(s_ann)}/{len(ent['signals'])}")
            result_entities.append({
                "entity": ent["entity"],
                "ports":   merge(p_ann, ent["ports"]),
                "signals": merge(s_ann, ent["signals"])})
        out_f.write_text(json.dumps(
            {"model": client.model, "module": m.name, "entities": result_entities}, indent=1))
        print(f"{'ok ' if not warned else 'OK*'} {m.name}: {len(parsed['entities'])} entities")
    except Exception as e:
        if "PerDay" in str(e):
            print(f"daily quota hit at {m.name} -- resume after reset"); break
        print(f"FAIL {m.name}: {e}")

ok  neorv32_boot_rom: 1 entities
ok  neorv32_bus: 6 entities
ok  neorv32_cache: 2 entities
ok  neorv32_cfs: 1 entities
ok  neorv32_clint: 4 entities
ok  neorv32_cpu: 1 entities
ok  neorv32_cpu_alu: 1 entities
ok  neorv32_cpu_control: 1 entities
ok  neorv32_cpu_counters: 2 entities
ok  neorv32_cpu_cp_bitmanip: 1 entities
ok  neorv32_cpu_cp_cfu: 1 entities
ok  neorv32_cpu_cp_cond: 1 entities
ok  neorv32_cpu_cp_crypto: 1 entities
ok  neorv32_cpu_cp_fpu: 3 entities
ok  neorv32_cpu_cp_muldiv: 1 entities
ok  neorv32_cpu_cp_shifter: 1 entities
ok  neorv32_cpu_decompressor: 1 entities
ok  neorv32_cpu_frontend: 1 entities
ok  neorv32_cpu_icc: 1 entities
ok  neorv32_cpu_lsu: 1 entities
ok  neorv32_cpu_pmp: 1 entities
ok  neorv32_cpu_regfile: 1 entities
ok  neorv32_crc: 1 entities
ok  neorv32_debug_auth: 1 entities
ok  neorv32_debug_dm: 1 entities
ok  neorv32_debug_dtm: 1 entities
ok  neorv32_dma: 1 entities
ok  neorv32_dmem: 1 entities
ok  neorv32_fifo: 1 entities
ok  neorv32_gpio: 1 entities
ok

In [45]:
import json
from pathlib import Path
for f in sorted(Path("stage_a_out/rtl_parsed").glob("*.json")):
    d = json.loads(f.read_text())
    unclear = sum(1 for e in d["entities"] for grp in ("ports","signals")
                  for x in e[grp] if x["function"] == "function unclear from RTL")
    n = sum(len(e[grp]) for e in d["entities"] for grp in ("ports","signals"))
    if unclear:
        print(f"{d['module']}: {unclear}/{n} unclear")
print("check done — 48 files")

check done — 48 files


# Stage B: Asset generation

In [4]:
# Cell G0 — Stage B prompts: compose core + ICL by concatenation; inventory ready modules
import importlib, json, time
from pathlib import Path
import prompts; importlib.reload(prompts)
from prompts import _ASSET_PRIMARY_CORE, ASSET_SECONDARY_SYSTEM
from icl_asset_examples import ICL_ASSET_EXAMPLES

ASSET_PRIMARY_SYSTEM = _ASSET_PRIMARY_CORE + "\n\n" + ICL_ASSET_EXAMPLES

SUM_DIR   = Path("stage_a_out/summaries")
PARSE_DIR = Path("stage_a_out/rtl_parsed")
PRIM_DIR  = Path("stage_a_out/assets_primary");   PRIM_DIR.mkdir(parents=True, exist_ok=True)
SEC_DIR   = Path("stage_a_out/assets_secondary"); SEC_DIR.mkdir(parents=True, exist_ok=True)
REJ_DIR   = Path("stage_a_out/assets_rejected");  REJ_DIR.mkdir(parents=True, exist_ok=True)

kept  = [m for m in mods if m.keep]
ready = [m for m in kept
         if (SUM_DIR / f"{m.name}.md").exists() and (PARSE_DIR / f"{m.name}.json").exists()]
print(f"kept {len(kept)} | ready (summary+parse present) {len(ready)}")
print(f"primary system prompt ~{len(ASSET_PRIMARY_SYSTEM)//4} tokens "
      f"(core {len(_ASSET_PRIMARY_CORE)//4} + ICL {len(ICL_ASSET_EXAMPLES)//4})")

kept 0 | ready (summary+parse present) 0
primary system prompt ~5914 tokens (core 998 + ICL 4915)


In [5]:
# Cell G1 — helpers: JSON-mode completion, input assembly, contract validators
import re
from llm import parse_json_array
import time as _t, random as _r
from google.genai import types

REQ_KEYS = {"Asset Name", "Asset RTL", "Entity", "Functionality",
            "Security Objective", "Justification"}
ENUM = {"Confidentiality", "Integrity", "Availability"}

def complete_json(client, system, user, max_tokens=8192):
    wait = 2.0
    for a in range(client.r):
        try:
            return client.c.models.generate_content(
                model=client.model, contents=user,
                config=types.GenerateContentConfig(
                    system_instruction=system, temperature=client.t,
                    response_mime_type="application/json",
                    max_output_tokens=max_tokens)).text
        except Exception as e:
            if "PerDay" in str(e): raise
            if a == client.r - 1: raise
            _t.sleep(wait + _r.uniform(0, 1)); wait *= 2

def load_parsed(m):
    return json.loads((PARSE_DIR / f"{m.name}.json").read_text())

def closed_set(parsed):
    """element name -> set of declaring entities (a name may exist in several)."""
    owner = {}
    for e in parsed["entities"]:
        for grp in ("ports", "signals"):
            for x in e[grp]:
                owner.setdefault(x["name"], set()).add(e["entity"])
    return owner

def elements_block(parsed):
    parts = []
    for e in parsed["entities"]:
        parts.append(f"Entity {e['entity']}:\nPARSED PORTS:\n"
                     + json.dumps(e["ports"], indent=1)
                     + "\nPARSED INTERNAL SIGNALS:\n"
                     + json.dumps(e["signals"], indent=1))
    return "\n\n".join(parts)

def primary_user(m, summary, parsed, rtl):
    return (f"TARGET IP MODULE: {m.name}\n\nTECHNICAL SUMMARY:\n{summary}\n\n"
            f"PARSED DESIGN ELEMENTS (closed set):\n{elements_block(parsed)}\n\n"
            f"RTL:\n{rtl}")


def _base(rtl):
    """Sub-element path -> base closed-set name.
    'csr.mtval' -> 'csr' ; 'sysinfo(0)' -> 'sysinfo' ; 'xtea.sum' -> 'xtea' ; 'pmpcfg' -> 'pmpcfg'."""
    return re.split(r"[.(]", str(rtl), 1)[0].strip()

def validate_primary(assets, owner):
    v = []
    if not assets:
        v.append("empty/unparseable array")
    seen = set()
    for a in assets:
        an = a.get("Asset RTL", "?")
        if set(a) != REQ_KEYS:
            v.append(f"{an}: keys {sorted(set(a) ^ REQ_KEYS)}")
        if a.get("Security Objective") not in ENUM:
            v.append(f"{an}: objective {a.get('Security Objective')!r}")
        base = _base(an)                              # resolve record field / array index to its base
        if base not in owner:
            v.append(f"{an}: base {base!r} not in closed set")
        elif a.get("Entity") not in owner[base]:
            v.append(f"{an}: entity {a.get('Entity')!r} does not declare {base!r}")
        if an in seen:                                # dedup on the FULL path, not the base
            v.append(f"{an}: duplicate primary")
        seen.add(an)
    return v

def validate_secondary(sec, primaries, owner):
    v = []
    want = [p["Asset RTL"] for p in primaries]
    got  = [s.get("Asset RTL") for s in sec]
    if got != want: v.append(f"order/set mismatch: got {got} want {want}")
    for s in sec:
        an = s.get("Asset RTL", "?")
        if set(s) != {"Asset RTL", "Secondary Assets"}: v.append(f"{an}: keys {sorted(s)}")
        for x in s.get("Secondary Assets", []):
            if _base(x) not in owner: v.append(f"{an}: secondary {x!r} not in closed set")
            if x == an:        v.append(f"{an}: lists itself as secondary")
    return v

In [6]:
# Cell G2 — primary asset generation (resumable; hard-reject on contract violations)
done = skipped = failed = rejected = 0
for m in ready:
    out_f, rej_f = PRIM_DIR / f"{m.name}.json", REJ_DIR / f"{m.name}.primary.json"
    if out_f.exists() or rej_f.exists():
        skipped += 1; continue
    parsed = load_parsed(m); owner = closed_set(parsed)
    if not owner:                           # package-only file: no structural elements
        print(f"skip {m.name}: empty closed set"); skipped += 1; continue
    summary = (SUM_DIR / f"{m.name}.md").read_text(encoding="utf-8")
    rtl     = (RTL_ROOT / m.file).read_text(encoding="utf-8", errors="ignore")
    try:
        raw    = complete_json(client, ASSET_PRIMARY_SYSTEM, primary_user(m, summary, parsed, rtl))
        assets = parse_json_array(raw)
        viol   = validate_primary(assets, owner)
        if viol:                            # non-conforming output = hard error, kept for inspection
            rej_f.write_text(json.dumps({"assets": assets, "violations": viol}, indent=1))
            rejected += 1
            print(f"REJ  {m.name}: {viol[:3]}{' ...' if len(viol) > 3 else ''}")
        else:
            out_f.write_text(json.dumps(
                {"model": client.model, "module": m.name, "assets": assets}, indent=1))
            done += 1
            print(f"ok   {m.name}: {len(assets)} primaries")
    except Exception as e:
        if "PerDay" in str(e):
            print(f"daily quota hit at {m.name} -- resume after reset"); break
        failed += 1; print(f"FAIL {m.name}: {e}")
    time.sleep(7)
print(f"\nprimary: {done} new | {skipped} skipped | {rejected} rejected | {failed} failed")


primary: 0 new | 0 skipped | 0 rejected | 0 failed


In [7]:
# Cell G6 — replication diagnostic vs repo's asset_list_neorv32_initial.json
# Two scopes: (A) IP-level coverage, (B) asset-level name overlap within matched IPs.
# NOTE: repo initial list is a REFERENCE POINT, not the golden set. The repo's own
# paper says its comparison baseline (Nath et al.) misses real assets; treat repo
# extras we lack as "review", not automatically "missed", and our extras as
# "possible over-generation OR genuine finds the repo omitted".
import json, re
from pathlib import Path

REPO_LIST = Path("LAsset-Security-Assets\\SoC\\Spec.+RTL\\asset_list_neorv32_initial.json")   # adjust to your copy's path
PRIM_DIR  = Path("stage_a_out/assets_primary")
REJ_DIR   = Path("stage_a_out/assets_rejected")

def _base(rtl):
    return re.split(r"[.(]", str(rtl), maxsplit=1)[0].strip()

# --- load repo reference: IP -> {full Asset RTL paths} and {base names} ---
repo = json.loads(REPO_LIST.read_text())
repo_ip = {}
for ip in repo:
    rtls = [a["Asset RTL"] for a in ip["Assets"]]
    repo_ip[ip["IP"]] = {"full": set(rtls), "base": {_base(r) for r in rtls},
                         "assets": ip["Assets"]}

# --- load our accepted primaries ---
ours_ip = {}
for f in sorted(PRIM_DIR.glob("*.json")):
    d = json.loads(f.read_text())
    rtls = [a["Asset RTL"] for a in d["assets"]]
    ours_ip[d["module"]] = {"full": set(rtls), "base": {_base(r) for r in rtls},
                            "assets": d["assets"]}

# rejected modules: present in our pipeline but no accepted output yet
rejected_mods = {f.name.split(".primary.json")[0] for f in REJ_DIR.glob("*.primary.json")}

# ============================================================
# SCOPE A — IP-level coverage
# ============================================================
repo_names, ours_names = set(repo_ip), set(ours_ip)
matched   = sorted(repo_names & ours_names)
we_missed = sorted(repo_names - ours_names)     # repo has it, we produced no accepted output
we_extra  = sorted(ours_names - repo_names)     # we output it, repo's list has no such IP

print("="*70, "\nSCOPE A — IP-LEVEL COVERAGE\n" + "="*70)
print(f"repo IPs: {len(repo_names)} | our accepted IPs: {len(ours_names)} | matched: {len(matched)}")

print(f"\n[MISSED] in repo, NOT in our accepted output ({len(we_missed)}):")
for n in we_missed:
    tag = "  <- REJECTED (sub-element collapse; rerun after fix)" if n in rejected_mods \
          else "  <- absent (dropped in Stage A? or no summary/parse?)"
    print(f"   {n}{tag}")

print(f"\n[EXTRA] in our output, NOT an IP in repo list ({len(we_extra)}):")
for n in we_extra:
    print(f"   {n}  ({len(ours_ip[n]['assets'])} primaries)  "
          f"<- repo may have pruned this module, or named it differently")

# ============================================================
# SCOPE B — asset-level name overlap within MATCHED IPs
# Compare on BASE names (record fields/array indices collapse to their base),
# because we and the repo may qualify differently (csr.mtval vs csr).
# ============================================================
print("\n" + "="*70, "\nSCOPE B — ASSET-LEVEL OVERLAP (matched IPs, base-name)\n" + "="*70)
print(f"{'IP':26s} {'ours':>4s} {'repo':>4s} {'hit':>4s}  {'miss (repo has, we lack)'}")
tot_repo = tot_hit = 0
missed_assets, extra_assets = {}, {}
for n in matched:
    ob, rb = ours_ip[n]["base"], repo_ip[n]["base"]
    hit  = ob & rb
    miss = rb - ob        # repo asset base we didn't produce
    extra= ob - rb        # our asset base the repo lacks
    tot_repo += len(rb); tot_hit += len(hit)
    if miss:  missed_assets[n] = sorted(miss)
    if extra: extra_assets[n]  = sorted(extra)
    flag = "" if not miss else "  " + ", ".join(sorted(miss)[:6]) + (" ..." if len(miss) > 6 else "")
    print(f"{n:26s} {len(ob):4d} {len(rb):4d} {len(hit):4d}{flag}")

recall = tot_hit / tot_repo if tot_repo else 0
print(f"\nbase-name recall over matched IPs: {tot_hit}/{tot_repo} = {recall:.1%}")
print("(NOT the paper's recall metric — no golden set, base-name proxy only)")

# ============================================================
# Detail dumps for prompt iteration
# ============================================================
print("\n" + "="*70, "\n[ASSET MISSES] repo-listed base names absent from our output\n" + "="*70)
for n, miss in sorted(missed_assets.items(), key=lambda kv: -len(kv[1])):
    # show repo's own Asset Name for each miss, so you can judge if it's a real asset
    lut = {_base(a["Asset RTL"]): a["Asset Name"] for a in repo_ip[n]["assets"]}
    print(f"\n{n}:")
    for b in miss:
        print(f"   - {b:22s} \"{lut.get(b,'?')}\"")

print("\n" + "="*70, "\n[ASSET EXTRAS] our base names the repo list doesn't have\n" + "="*70)
print("(over-generation candidates — OR genuine finds the repo omitted; judge each)")
for n, extra in sorted(extra_assets.items(), key=lambda kv: -len(kv[1])):
    lut = {_base(a["Asset RTL"]): a["Asset Name"] for a in ours_ip[n]["assets"]}
    print(f"\n{n}:")
    for b in extra:
        print(f"   + {b:22s} \"{lut.get(b,'?')}\"")

# machine-readable dump for scripting the next prompt iteration
Path("stage_a_out/compare_vs_repo.json").write_text(json.dumps({
    "ip_missed": we_missed, "ip_extra": we_extra, "matched": matched,
    "asset_missed": missed_assets, "asset_extra": extra_assets,
    "base_recall": recall}, indent=1))
print("\nwrote stage_a_out/compare_vs_repo.json")

SCOPE A — IP-LEVEL COVERAGE
repo IPs: 41 | our accepted IPs: 0 | matched: 0

[MISSED] in repo, NOT in our accepted output (41):
   neorv32_bus  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cache  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_alu  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_control  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_counters  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_bitmanip  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_cfu  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_cond  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_crypto  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_fpu  <- absent (dropped in Stage A? or no summary/parse?)
   neorv32_cpu_cp_muldiv  <- absent (dropped i